# AgentState와 context_schema

이 노트북은 Python 3.12 새 커널에서 독립 실행합니다.


## 패키지 설치


In [1]:
# 노트북이 사용하는 패키지 버전을 새 커널 환경에 설치합니다.
%pip install -q langchain==1.4.0 langchain-openai==1.6.0 langgraph==1.2.11 python-dotenv==1.2.3 pillow

Note: you may need to restart the kernel to use updated packages.


## 환경 준비


In [2]:
# 공개하지 않은 API 키는 존재 여부만 확인하고 고정한 패키지 버전을 출력합니다.
import os
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").is_file():
        load_dotenv(folder / ".env")
        break
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 필요합니다."
print("langchain", version("langchain"))
print("langgraph", version("langgraph"))
print("API 키 존재 여부 확인 완료")

langchain 1.4.0
langgraph 1.2.11
API 키 존재 여부 확인 완료


## 상태와 실행 컨텍스트 정의

상태(state)는 그래프 단계 사이에 남고 체크포인터가 저장할 수 있는 실행 데이터입니다. `SupportState(AgentState)`의 `request_tag`는 입력 상태에 들어가고 최종 결과에도 남습니다. 실행 컨텍스트(context)는 한 번의 호출 동안 노드와 도구가 참고하지만 상태 채널로 자동 저장되지 않는 의존 정보입니다. `UserContext`를 데이터 클래스로 선언한 이유는 `user_id`와 `access_level`이라는 입력 형식을 호출 전에 고정하기 위해서입니다.

`ToolRuntime[UserContext]`는 모델이 채우는 도구 인자가 아닙니다. LangChain 실행기가 `current_access`를 호출할 때 런타임 객체를 주입하고, 함수는 `runtime.context`에서 호출자가 건넨 컨텍스트를 읽습니다. `agent.invoke(state, context=UserContext(...))`에서 첫 번째 인자는 누적 상태이고 `context=`는 실행 의존 정보입니다. 두 값을 분리하면 모델에게 권한 값을 도구 인자로 생성하게 하지 않아도 됩니다.


In [3]:
# request_tag는 AgentState에, 사용자 권한은 ToolRuntime이 읽는 context에 둡니다.
from dataclasses import dataclass
from langchain.agents import AgentState, create_agent
from langchain.tools import tool, ToolRuntime
class SupportState(AgentState):
    request_tag: str
@dataclass
class UserContext:
    user_id: str
    access_level: str
@tool
def current_access(runtime: ToolRuntime[UserContext]) -> str:
    """현재 사용자의 접근 수준을 확인합니다."""
    return f"{runtime.context.user_id}:{runtime.context.access_level}"

## 컨텍스트를 전달해 실행


In [4]:
# invoke의 상태 입력과 context 인자를 분리해 current_access 도구를 실행합니다.
agent = create_agent("openai:gpt-5.4-mini", [current_access], system_prompt="접근 수준 질문에는 도구를 반드시 사용하세요.", state_schema=SupportState, context_schema=UserContext)
result = agent.invoke({"messages": [{"role":"user","content":"제 접근 수준을 확인해 주세요."}], "request_tag":"lesson-03"}, context=UserContext("student-7", "read-only"))
print("state keys:", sorted(result.keys()))
print("request_tag:", result["request_tag"])
print("answer:", result["messages"][-1].text)
assert result["request_tag"] == "lesson-03" and "read-only" in result["messages"][-1].text

state keys: ['messages', 'request_tag']
request_tag: lesson-03
answer: 현재 접근 수준은 **student-7:read-only** 입니다.


## 상태와 컨텍스트 구분

쇼핑 상담에서는 장바구니 항목과 대화 메시지를 상태에 두고, 데이터베이스 연결이나 현재 사용자의 권한 객체를 컨텍스트에 둘 수 있습니다. 같은 권한을 다음 날 다시 써야 한다면 컨텍스트만으로는 충분하지 않으며 별도 저장소에서 다시 읽거나 상태에 필요한 식별자를 남겨야 합니다.

이번 출력에서 `request_tag`가 남고 `context` 키가 없는 사실은 자동 저장 여부만 보여 줍니다. 도구가 컨텍스트 값을 답 문자열에 포함했기 때문에 `read-only`는 메시지에는 나타납니다. 이것을 컨텍스트 객체 전체가 상태에 저장됐다는 뜻으로 해석하면 안 됩니다.


In [5]:
# 최종 상태 키를 검사해 context가 AgentState에 자동 저장되지 않음을 확인합니다.
print("AgentState는 그래프 실행 결과에 남음:", result["request_tag"])
print("context는 도구 실행 중 읽고 결과 상태의 키로 자동 저장되지 않음:", "context" in result)
assert "context" not in result

AgentState는 그래프 실행 결과에 남음: lesson-03
context는 도구 실행 중 읽고 결과 상태의 키로 자동 저장되지 않음: False
